# Split and Baseline

**Notebook 2 of 6.** Notebook 01 ended on a problem: a model is only worth what it does on
penguins it has not seen, yet every penguin with a known sex is one we would like to learn
from. This notebook solves it by holding some penguins back as a **test set**.

It then asks the question that comes before any model: **what score would a model get
without learning anything?** That number, the **baseline**, is what every model after it
has to beat. By the end you will have two baselines, one very simple rule, and a "model"
that scores 100 percent and has learned nothing at all.

The libraries, then the labeled penguins from notebook 01.

In [ ]:
# data analysis
import numpy as np
import pandas as pd

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# machine learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

sns.set_style("whitegrid")

The features are the four measurements. The text columns `species` and `island` are left
out for now: a model works with numbers, and turning categories into numbers is a step
notebook 06 takes.

In [ ]:
df_penguins = pd.read_csv("data/penguins.csv")
labeled = df_penguins.dropna(subset=["sex"])

features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = labeled[features]
y = labeled["sex"]

X.shape, y.shape

## 1. Why hold data back

Think of an exam. A student who has seen the exact questions in advance can score full
marks by remembering the answers, and the score then says nothing about whether they
understood anything. To measure understanding, the exam has to use questions the student
has **not** practiced on.

A model is in the same position. So before training anything, the labeled data is split
into two parts:

- The **training set**, which the model learns from. It may be looked at, plotted, and
  used for any decision you like.
- The **test set**, which is put aside and used **once, at the end**, to measure how well
  the finished model does on penguins it has never seen.

The test set stands in for the future. It is only a fair stand-in if nothing about it
influenced the model: not its labels, not its values, not even a glance at a plot of it.

## 2. The split

scikit-learn does the split in one line.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("train:", X_train.shape, y_train.shape)
print("test: ", X_test.shape, y_test.shape)

Four results come back, always in this order: the training features, the test features,
the training target, the test target. The arguments:

- `test_size=0.25` puts a quarter of the rows in the test set, here 84 of 333.
- `random_state=42` fixes the shuffle. The rows are assigned at random, and without a fixed
  seed you would get a different split, and different results, on every run.
- `stratify=y` keeps the share of each class the same in both parts. Without it, a small
  test set can end up with noticeably more of one class by chance.

In [ ]:
pd.DataFrame({
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).round(3)

> **Exercise:**
>
> Split the same data again with `test_size=0.2` and `random_state=0`, but **without**
> `stratify`. Compare the share of each class in the two parts with the table above.

In [ ]:
# your code: split without stratify, then compare the class shares

<details><summary>
Click here to compare your answer...
</summary>

The shares drift apart, and by more than you might expect: this test set is about 61
percent female, the training set about 47 percent. The full data is almost exactly 50/50,
so the whole difference is the luck of the shuffle. On a target where one class is rare,
the same luck can leave a test set with hardly any of the rare class at all. Stratifying
costs nothing, so the rest of this repository always does it.
</details>

## 3. Measuring a prediction: accuracy

A prediction is compared with the truth one row at a time. The simplest summary is
**accuracy**: the share of rows predicted correctly.

$$\text{accuracy} = \frac{\text{number of correct predictions}}{\text{number of predictions}}$$

In code, comparing two columns gives `True` or `False` for each row, and the mean of those
is the share of `True`:

In [ ]:
# three predictions against three true labels: two are right
truth = pd.Series(["MALE", "FEMALE", "MALE"])
guess = pd.Series(["MALE", "MALE", "MALE"])

print((guess == truth).mean())
print(accuracy_score(truth, guess))

Accuracy is not the only way to score a classifier, and not always the right one. For now
it is the one number used throughout; the other scores get their own lesson.

## 4. The first baseline: always guess the majority

The laziest possible "model" ignores the features and predicts the most common class in
the training set for every penguin.

In [ ]:
majority = y_train.mode()[0]
print("majority class in the training set:", majority)

y_pred_majority = np.full(len(y_test), majority)
print("test accuracy:", accuracy_score(y_test, y_pred_majority))

**50 percent.** The test set holds 42 females and 42 males, so a constant guess is right
exactly half the time. This is the floor: a model that does not beat 50 percent here has
learned nothing a coin could not do.

Note that the majority class was read off the **training** set. Even the laziest model is
built from training data only.

## 5. A model that remembers everything and learns nothing

Now the opposite extreme. This "model" stores every training penguin with its label, like
a dictionary. Asked about a penguin it has seen, it returns the stored label. Asked about
any other penguin, it falls back to the majority guess.

In [ ]:
def fit_memorizer(X_train, y_train):
    """Remember the label of every training row.

    Parameters
    ----------
    X_train : pandas.DataFrame
        The training features.
    y_train : pandas.Series
        The training labels, in the same row order.

    Returns
    -------
    dict
        Maps each row of features, as a tuple, to its label.
    """
    return {tuple(row): label for row, label in zip(X_train.to_numpy(), y_train)}


def predict_memorizer(memory, X, fallback):
    """Look each row up in memory, and guess `fallback` for rows never seen.

    Parameters
    ----------
    memory : dict
        The output of `fit_memorizer`.
    X : pandas.DataFrame
        The rows to predict, with the same columns as the training features.
    fallback : str
        The label to predict for a row that is not in memory.

    Returns
    -------
    list of str
        One predicted label per row.
    """
    return [memory.get(tuple(row), fallback) for row in X.to_numpy()]

In [ ]:
memory = fit_memorizer(X_train, y_train)

train_acc = accuracy_score(y_train, predict_memorizer(memory, X_train, majority))
test_acc = accuracy_score(y_test, predict_memorizer(memory, X_test, majority))

print(f"train accuracy: {train_acc:.3f}")
print(f"test accuracy:  {test_acc:.3f}")

**100 percent on the training set, 50 percent on the test set.** No test penguin has
exactly the measurements of a training penguin, so every single test prediction is the
fallback guess, and the memorizer is no better than the majority baseline.

Two lessons in one result:

- **A score on the training set says nothing about generalization.** The memorizer's 100
  percent is an exam passed with the answers in hand.
- **The gap between training and test score is the warning sign.** A model that does far
  better on data it has seen than on data it has not has memorized rather than learned.
  This is called **overfitting**, and real models do it in subtler ways than this one.

## 6. The second baseline: one rule

Male penguins tend to be heavier. A rule a person could write down: **predict male when
the body mass is above some threshold.** A first choice for the threshold is the median
mass of the training penguins.

In [ ]:
def predict_by_threshold(values, threshold):
    """Predict MALE where a measurement is above `threshold`, FEMALE otherwise.

    Parameters
    ----------
    values : pandas.Series
        One measurement per penguin.
    threshold : float
        Values above it are predicted MALE.

    Returns
    -------
    numpy.ndarray
        One predicted label per penguin.
    """
    return np.where(values > threshold, "MALE", "FEMALE")

In [ ]:
threshold = X_train["body_mass_g"].median()
print("threshold:", threshold, "g")

train_acc = accuracy_score(y_train, predict_by_threshold(X_train["body_mass_g"], threshold))
test_acc = accuracy_score(y_test, predict_by_threshold(X_test["body_mass_g"], threshold))

print(f"train accuracy: {train_acc:.3f}")
print(f"test accuracy:  {test_acc:.3f}")

Around two thirds right on both sets. That is clearly better than a coin, and the train
and test scores are close, so whatever this rule knows, it knows about penguins in general
and not about these particular 249.

The median was a guess, though. The plot shows how the masses of the two sexes overlap in
the training set:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(x=X_train["body_mass_g"], hue=y_train, bins=25, ax=ax)
ax.axvline(threshold, color="black", linestyle="--")  # the median threshold
ax.set_xlabel("body mass (g)")
ax.set_title("Body mass of the training penguins, by sex");

The dashed line is the median. Left of it, mostly females; right of it, a mix. The second
hump of heavy penguins is the Gentoo species, whose **females outweigh most males of the
other two species**. No single mass threshold can be right for all three species at once,
which is a first hint that a model will do better when it sees more than one column.

> **Exercise:**
>
> Find a better threshold. Try every value from 3,000 g to 6,000 g in steps of 50 g, and
> keep the one with the highest **training** accuracy. Then evaluate that single
> threshold on the test set, once.

<details><summary>
Click here for a hint...
</summary>

`range(3000, 6001, 50)` gives the candidates. For each one, compute the training accuracy
with `predict_by_threshold` and `accuracy_score`, and remember the best. Collecting the
results in a dictionary and taking `max(results, key=results.get)` is one way.
</details>

In [ ]:
# your code: choose the threshold on the training set, then test it once

<details><summary>
Click here to compare your answer...
</summary>

The best threshold on the training set is **3,700 g**, with a training accuracy of about
73 percent and a test accuracy of about 71 percent. Choosing the threshold was a small act
of learning: the rule now comes from the data rather than from a guess, and it
generalizes, since the test score is close to the training score.
</details>

**Why choose on the training set only?** It would have been easy to try all the thresholds
on the test set and keep the best one there. The test score would have gone up, and it
would have stopped meaning anything: the test set would then have been used to *build* the
rule, and would no longer stand in for penguins the rule has never seen. Letting the test
set influence the model is called **test leakage**, and notebook 05 shows how quietly it
can happen.

Body mass is only one of four measurements. The next cell wraps the search you just wrote
into a function that works for any column. The candidate thresholds are the distinct
values of the training column, since bill depth is measured in tens of millimeters and
body mass in thousands of grams.

In [ ]:
def best_threshold(feature):
    """Search the training values of `feature` for the most accurate threshold.

    Parameters
    ----------
    feature : str
        A column of `X_train`.

    Returns
    -------
    tuple of (float, float)
        The best threshold and its training accuracy.
    """
    results = {}
    for candidate in np.unique(X_train[feature]):
        pred = predict_by_threshold(X_train[feature], candidate)
        results[candidate] = accuracy_score(y_train, pred)
    best = max(results, key=results.get)
    return best, results[best]


best_threshold("body_mass_g")

> **Exercise:**
>
> Use `best_threshold` on all four features. Collect, for each feature, its threshold, its
> training accuracy and its test accuracy in one table. **Which feature would you choose,
> and which column of the table do you base that choice on?**

<details><summary>
Click here for a hint...
</summary>

Loop over `features`, append one dictionary per feature to a list, and turn the list into
a table with `pd.DataFrame(rows)`.
</details>

In [ ]:
# your code: a table of all four features

<details><summary>
Click here to compare your answer...
</summary>

**Body mass**, chosen on the **training** column, where it scores best at about 73
percent. The test column tempts you toward bill length, which happens to do best there
at 75 percent. Picking by that column would be the same mistake as tuning the threshold
on the test set: the test set would have helped make the choice.

The table also shows how noisy a test set of 84 penguins is. Bill depth is second best on
the training set and worst but one on the test set. With so few rows, a handful of
penguins decides the order.
</details>

## 7. One split is one sample

The test score depends on which penguins happened to land in the test set. The next cell
repeats the median rule from section 6 on ten different splits, each with its own
training median, and records the test accuracy of each.

In [ ]:
scores = []
for seed in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y
    )
    threshold = X_tr["body_mass_g"].median()
    scores.append(accuracy_score(y_te, predict_by_threshold(X_te["body_mass_g"], threshold)))

print(np.round(scores, 3))
print(f"lowest {min(scores):.3f}, highest {max(scores):.3f}")

The same rule scores anywhere from about 57 to 70 percent, depending only on the split. A
single test score is a measurement with an error bar you cannot see, just like the sample
average in a hypothesis test. Two models whose test scores differ by a couple of points
may well be equally good. **Cross-validation**, later in the program, measures a model
on many splits at once for exactly this reason.

## Summary

| Approach | Train accuracy | Test accuracy | What it shows |
| :-- | --: | --: | :-- |
| Always the majority | 51% | 50% | the floor any model must beat |
| Memorizer | 100% | 50% | a perfect training score can mean nothing |
| Mass above the median | 64% | 67% | a simple rule that generalizes |
| Mass above 3,700 g | 73% | 71% | a rule learned from the training set |

- The **test set** is held back and used once, at the end. **Nothing about it may shape
  the model.** Using it to choose is **test leakage**.
- `train_test_split` with `stratify` and a fixed `random_state` gives a fair, repeatable
  split.
- A **baseline** is the score without real learning. A model is judged by how far it
  beats it.
- A large gap between training and test score is **overfitting**: memorizing instead of
  learning.

Next, in `03_how_logistic_regression_works.ipynb`: instead of a hard threshold, a model
that gives each penguin a **probability** of being male, and chooses its own boundary.